In [1]:
from google.colab import drive
from google.colab import auth

# 1. Gắn Google Drive vào Colab
drive.mount('/content/drive')

# 2. Xác thực tài khoản Google Cloud (Sẽ có popup yêu cầu cấp quyền)
auth.authenticate_user()

# 3. Cấu hình Project ID
# THAY THẾ 'your-gcp-project-id' BẰNG ID PROJECT THẬT CỦA BẠN TRÊN GCP
PROJECT_ID = 'mining-data-494820'
!gcloud config set project {PROJECT_ID}

Mounted at /content/drive
[environment: untagged] Read more to tag: g.co/cloud/project-env-tag.
Updated property [core/project].


To take a quick anonymous survey, run:
  $ gcloud survey



In [2]:
import os
import subprocess
import glob

# 1. Thư mục chứa các file .jsonl.gz và .jsonl trên Google Drive
DRIVE_DIR = '/content/drive/MyDrive/amazon_gpc'

# 2. GCS Bucket đích
GCS_DESTINATION = 'gs://mining-data-2/raw_data/amazon_gpc/'

def stream_unzip_to_gcs():
    """Đọc file từ Drive và đẩy lên GCS (giải nén nếu là .gz)."""

    # Tìm tất cả file .gz và .jsonl
    gz_files = glob.glob(os.path.join(DRIVE_DIR, '**', '*.gz'), recursive=True)
    jsonl_files = glob.glob(os.path.join(DRIVE_DIR, '**', '*.jsonl'), recursive=True)
    all_files = sorted(list(set(gz_files + jsonl_files)))

    if not all_files:
        print(f"Không tìm thấy file .gz hoặc .jsonl nào tại: {DRIVE_DIR}")
        return

    # In danh sách file để kiểm tra bằng mắt
    print(f"--- DANH SÁCH FILE TÌM THẤY ({len(all_files)} file) ---")
    for f in all_files:
        print(f"File: {os.path.basename(f)}")
    print("-" * 50)

    print("Bắt đầu quy trình đẩy dữ liệu lên GCS...")

    for i, file_path in enumerate(all_files, 1):
        file_name = os.path.basename(file_path)
        is_gz = file_name.endswith('.gz')

        # Tên file đích trên GCS (loại bỏ đuôi .gz nếu có)
        target_name = file_name[:-3] if is_gz else file_name
        gcs_target_path = f"{GCS_DESTINATION.rstrip('/')}/{target_name}"

        print(f"[{i}/{len(all_files)}] ĐANG XỬ LÝ: {file_name}")

        try:
            if is_gz:
                print(f"   -> Đang giải nén qua RAM và stream lên: {gcs_target_path}")
                zcat_process = subprocess.Popen(["zcat", file_path], stdout=subprocess.PIPE)
                gsutil_process = subprocess.Popen(["gsutil", "cp", "-", gcs_target_path], stdin=zcat_process.stdout)

                zcat_process.stdout.close()
                gsutil_process.communicate()

                if gsutil_process.returncode != 0:
                    raise Exception(f"Lỗi gsutil khi streaming: {gsutil_process.returncode}")

                zcat_process.wait()
                if zcat_process.returncode != 0:
                    raise Exception(f"Lỗi zcat (có thể file hỏng): {zcat_process.returncode}")
            else:
                print(f"   -> Đang tải trực tiếp lên: {gcs_target_path}")
                result = subprocess.run(["gsutil", "cp", file_path, gcs_target_path], capture_output=True, text=True)
                if result.returncode != 0:
                    raise Exception(f"Lỗi gsutil khi upload trực tiếp: {result.stderr}")

            print(f"   => THÀNH CÔNG: Đã tải xong {target_name}")

        except Exception as e:
            print(f"   !!! THÔNG BÁO LỖI: {str(e)}")

    print("-" * 50)
    print("🎉 TẤT CẢ DỮ LIỆU ĐÃ SẴN SÀNG CHO PYSPARK! 🎉")

In [3]:
stream_unzip_to_gcs()

--- DANH SÁCH FILE TÌM THẤY (14 file) ---
File: Cell_Phones_and_Accessories.jsonl.gz
File: Electronics.jsonl.gz
File: Industrial_and_Scientific.jsonl.gz
File: Movies_and_TV.jsonl.gz
File: dmx_metadatas.jsonl
File: dmx_reviews.jsonl
File: meta_Cell_Phones_and_Accessories.jsonl.gz
File: meta_Electronics.jsonl.gz
File: meta_Industrial_and_Scientific.jsonl.gz
File: meta_Movies_and_TV.jsonl.gz
File: metadatas_amazon.jsonl
File: reviews_amazon.jsonl
File: tgdd_metadatas.jsonl
File: tgdd_reviews.jsonl
--------------------------------------------------
Bắt đầu quy trình đẩy dữ liệu lên GCS...
[1/14] ĐANG XỬ LÝ: Cell_Phones_and_Accessories.jsonl.gz
   -> Đang giải nén qua RAM và stream lên: gs://mining-data-2/raw_data/amazon_gpc/Cell_Phones_and_Accessories.jsonl
   => THÀNH CÔNG: Đã tải xong Cell_Phones_and_Accessories.jsonl
[2/14] ĐANG XỬ LÝ: Electronics.jsonl.gz
   -> Đang giải nén qua RAM và stream lên: gs://mining-data-2/raw_data/amazon_gpc/Electronics.jsonl
   => THÀNH CÔNG: Đã tải xong El